# Structure Formation and LSS

This notebook demonstrates Lagrangian Perturbation Theory, Effective Field Theory models, and Gaussian Random Field simulations.

## Lagrangian Perturbation Theory (LPT) Solver

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import toolscosmo as tcm
import tools21cm as t2c

grid_size = 128 #512 #
box_size  = 500 #Mpc/h 
param = tcm.par()
param.file.ps = 'CAMB' # 'CLASS' 'CLASSemu'
param.file.ps = tcm.get_Plin(param)
iSeed = 314159265

z1 = 149.
z2 = 9.

delta_lin = tcm.generate_gaussian_random_field(grid_size, box_size, param=param, random_seed=iSeed)['delta_lin']
particle_pos_z1 = tcm.generate_initial_condition_positions(grid_size, box_size, z1, param, LPT=2, delta_lin=delta_lin)['positions']
particle_pos_z2 = tcm.generate_initial_condition_positions(grid_size, box_size, z2, param, LPT=2, delta_lin=delta_lin)['positions']
delta_z1 = tcm.particles_on_grid(particle_pos_z1, grid_size, box_size)
delta_z2 = tcm.particles_on_grid(particle_pos_z2, grid_size, box_size)

fig, axs = plt.subplots(1,3,figsize=(12,4))
axs[0].set_title('Gaussian Random Field')
xx = np.linspace(0,box_size,grid_size)
axs[0].pcolor(xx, xx, delta_lin[:,:,grid_size//2])
xx = np.linspace(0,box_size,grid_size)
axs[1].set_title(f'$z={z1}$')
axs[1].pcolor(xx, xx, delta_z1[:,:,grid_size//2])
axs[2].set_title(f'$z={z2}$')
axs[2].pcolor(xx, xx, delta_z2[:,:,grid_size//2])
for ax in axs.flatten():
    ax.set_xlabel('[Mpc/$h$]')
    ax.set_ylabel('[Mpc/$h$]')
plt.tight_layout()
plt.show()

ps0 = t2c.power_spectrum_1d(delta_lin, kbins=20, box_dims=box_size)
ps1 = t2c.power_spectrum_1d(delta_z1, kbins=20, box_dims=box_size)
ps2 = t2c.power_spectrum_1d(delta_z2, kbins=20, box_dims=box_size)
p0, k0 = ps0
p1, k1 = ps1[0]/tcm.growth_factor(z1, param)**2, ps1[1]
p2, k2 = ps2[0]/tcm.growth_factor(z2, param)**2, ps2[1]

fig, ax = plt.subplots(1,1,figsize=(5,4))
kk, pp = param.file.ps['k'], param.file.ps['P']
ax.loglog(kk, pp, c='k', ls='-', label='$P_\mathrm{{lin}}$')
ax.loglog(k0, p0, c='r', ls='--', label='$P^{{grid}}_\mathrm{{lin}}$')
ax.loglog(k1, p1*(grid_size/box_size)**-1, c='C0', ls='-.', label=f'$z={z1}$')
ax.loglog(k2, p2*(grid_size/box_size)**-1, c='C1', ls=':', label=f'$z={z2}$')
ax.set_xlabel('[$h$/Mpc]')
ax.set_ylabel('$P(k)$')
ax.axis([8e-3,3,3,4e4])
ax.legend()
plt.tight_layout()
plt.show()

## Effective Field Theory (EFT) Model

In [ ]:
import numpy as np
from numpy import sqrt
from time import time
import matplotlib.pyplot as plt
import os, sys
from tqdm import tqdm
from scipy.interpolate import interp1d

import toolscosmo




exit()

par = toolscosmo.par()
plin = toolscosmo.get_Plin(par)
par.file.ps = plin
eft = toolscosmo.EFTformalism_Anastasiou2024(par, save_folder='work/IntegralMatrices', verbose=False)
eft.compute_tracer_intergrals()
ps_fit = eft.fit_P_linear(plin=plin)

fig, ax = plt.subplots(1,1,figsize=(7,5))
k, P_L = plin['k'], plin['P']
ax.loglog(k, P_L, lw=3, ls='-', label='$P_\mathrm{lin}$')
PlinSpline = interp1d(plin['k'], plin['P'], kind='cubic', fill_value='extrapolate')
k_sample = np.logspace(-3, np.log10(2), 150)
ax.loglog(k_sample, PlinSpline(k_sample), lw=3, ls='--', label='Spline')
P_Lfit = ps_fit['Plin_fit']
ax.loglog(k, P_Lfit, lw=3, ls=':', label='fit')
ax.set_xlabel(f'$k [h/\mathrm{{Mpc}}]$')
ax.set_ylabel(f'$P(k)$')
plt.tight_layout()
plt.show()

k_mod = np.logspace(-3, np.log10(2), 150)
redshift = 9
b1 = 1.0  
b2 = 0.5
bG2 = -1
bGamma3 = 0.0 #0.2
R2 = 0.1
Pshot = 0.0 #1
cs2 = 0.00 #0.01

fig, axs = plt.subplots(2,4,figsize=(15,5))
cmap = plt.get_cmap('jet')
axs[0,0].set_title('Redshift')
param_array = np.linspace(6,12,10)
norm = plt.Normalize(vmin=param_array.min(), vmax=param_array.max())
sm = plt.cm.ScalarMappable(cmap=cmap, norm=norm)
sm.set_array([])
for i0 in param_array:
    pmod = eft.model_P_tracer(k_mod, b1, b2, bG2, bGamma3, R2, Pshot, cs2, i0)
    axs[0,0].loglog(k_mod, pmod*k_mod**3/2/np.pi**2, color=cmap(norm(i0)))
cbar = fig.colorbar(sm, ax=axs[0,0], orientation='vertical', pad=0.01)
axs[0,1].set_title('$b_1$')
param_array = np.linspace(-20,20,10)
norm = plt.Normalize(vmin=param_array.min(), vmax=param_array.max())
sm = plt.cm.ScalarMappable(cmap=cmap, norm=norm)
sm.set_array([])
for i0 in param_array:
    pmod = eft.model_P_tracer(k_mod, i0, b2, bG2, bGamma3, R2, Pshot, cs2, redshift)
    axs[0,1].loglog(k_mod, pmod*k_mod**3/2/np.pi**2, color=cmap(norm(i0)))
cbar = fig.colorbar(sm, ax=axs[0,1], orientation='vertical', pad=0.01)
axs[0,2].set_title('$b_2$')
param_array = np.linspace(-5,5,10)
norm = plt.Normalize(vmin=param_array.min(), vmax=param_array.max())
sm = plt.cm.ScalarMappable(cmap=cmap, norm=norm)
sm.set_array([])
for i0 in param_array:
    pmod = eft.model_P_tracer(k_mod, b1, i0, bG2, bGamma3, R2, Pshot, cs2, redshift)
    axs[0,2].loglog(k_mod, pmod*k_mod**3/2/np.pi**2, color=cmap(norm(i0)))
cbar = fig.colorbar(sm, ax=axs[0,2], orientation='vertical', pad=0.01)
axs[0,3].set_title('$b_{G_2}$')
param_array = np.linspace(-20,20,10)
norm = plt.Normalize(vmin=param_array.min(), vmax=param_array.max())
sm = plt.cm.ScalarMappable(cmap=cmap, norm=norm)
sm.set_array([])
for i0 in param_array:
    pmod = eft.model_P_tracer(k_mod, b1, b2, i0, bGamma3, R2, Pshot, cs2, redshift)
    axs[0,3].loglog(k_mod, pmod*k_mod**3/2/np.pi**2, color=cmap(norm(i0)))
cbar = fig.colorbar(sm, ax=axs[0,3], orientation='vertical', pad=0.01)
axs[1,0].set_title('$b_{\Gamma_3}$')
param_array = np.linspace(-20,20,10)
norm = plt.Normalize(vmin=param_array.min(), vmax=param_array.max())
sm = plt.cm.ScalarMappable(cmap=cmap, norm=norm)
sm.set_array([])
for i0 in param_array:
    pmod = eft.model_P_tracer(k_mod, b1, b2, bG2, i0, R2, Pshot, cs2, redshift)
    axs[1,0].loglog(k_mod, pmod*k_mod**3/2/np.pi**2, color=cmap(norm(i0)))
cbar = fig.colorbar(sm, ax=axs[1,0], orientation='vertical', pad=0.01)
axs[1,1].set_title('$R2$')
param_array = np.linspace(-20,20,10)
norm = plt.Normalize(vmin=param_array.min(), vmax=param_array.max())
sm = plt.cm.ScalarMappable(cmap=cmap, norm=norm)
sm.set_array([])
for i0 in param_array:
    pmod = eft.model_P_tracer(k_mod, b1, b2, bG2, bGamma3, i0, Pshot, cs2, redshift)
    axs[1,1].loglog(k_mod, pmod*k_mod**3/2/np.pi**2, color=cmap(norm(i0)))
cbar = fig.colorbar(sm, ax=axs[1,1], orientation='vertical', pad=0.01)
axs[1,2].set_title('$P_{shot}$')
param_array = np.linspace(-20,20,10)
norm = plt.Normalize(vmin=param_array.min(), vmax=param_array.max())
sm = plt.cm.ScalarMappable(cmap=cmap, norm=norm)
sm.set_array([])
for i0 in param_array:
    pmod = eft.model_P_tracer(k_mod, b1, b2, bG2, bGamma3, R2, i0, cs2, redshift)
    axs[1,2].loglog(k_mod, pmod*k_mod**3/2/np.pi**2, color=cmap(norm(i0)))
cbar = fig.colorbar(sm, ax=axs[1,2], orientation='vertical', pad=0.01)
axs[1,3].set_title('$c^2_{s}$')
param_array = np.linspace(-20,20,10)
norm = plt.Normalize(vmin=param_array.min(), vmax=param_array.max())
sm = plt.cm.ScalarMappable(cmap=cmap, norm=norm)
sm.set_array([])
for i0 in param_array:
    pmod = eft.model_P_tracer(k_mod, b1, b2, bG2, bGamma3, R2, Pshot, i0, redshift)
    axs[1,3].loglog(k_mod, pmod*k_mod**3/2/np.pi**2, color=cmap(norm(i0)))
cbar = fig.colorbar(sm, ax=axs[1,3], orientation='vertical', pad=0.01)
for ax in axs.flatten():
    ax.set_xlabel(f'$k [h/\mathrm{{Mpc}}]$')
    ax.set_ylabel(f'$P(k)$')
plt.tight_layout()
plt.show()


##### W. Qin et al., Phys. Rev. D106, 123506 (2022)

par = toolscosmo.par()
plin = toolscosmo.get_Plin(par)
par.file.ps = plin
eft = toolscosmo.EFTformalism_Qin2022(par, save_folder='work/IntegralMatrices', verbose=False)
eft.compute_tracer_intergrals()
ps_fit = eft.fit_P_linear(plin=plin)

k_mod = np.logspace(-3, np.log10(2), 150)
redshift = 9
b1 = 1.0  
b2 = 0.5
bG2 = -1
R2 = 0.1

fig, axs = plt.subplots(2,2,figsize=(9,7))
cmap = plt.get_cmap('jet')
axs[0,0].set_title('$b_1$')
param_array = np.linspace(-20,20,10)
norm = plt.Normalize(vmin=param_array.min(), vmax=param_array.max())
sm = plt.cm.ScalarMappable(cmap=cmap, norm=norm)
sm.set_array([])
for i0 in param_array:
    pmod = eft.model_P_tracer(k_mod, i0, b2, bG2, R2, redshift)
    axs[0,0].loglog(k_mod, pmod*k_mod**3/2/np.pi**2, color=cmap(norm(i0)))
cbar = fig.colorbar(sm, ax=axs[0,0], orientation='vertical', pad=0.01)
axs[0,1].set_title('$b_2$')
param_array = np.linspace(-5,5,10)
norm = plt.Normalize(vmin=param_array.min(), vmax=param_array.max())
sm = plt.cm.ScalarMappable(cmap=cmap, norm=norm)
sm.set_array([])
for i0 in param_array:
    pmod = eft.model_P_tracer(k_mod, b1, i0, bG2, R2, redshift)
    axs[0,1].loglog(k_mod, pmod*k_mod**3/2/np.pi**2, color=cmap(norm(i0)))
cbar = fig.colorbar(sm, ax=axs[0,1], orientation='vertical', pad=0.01)
axs[1,0].set_title('$b_{G_2}$')
param_array = np.linspace(-20,20,10)
norm = plt.Normalize(vmin=param_array.min(), vmax=param_array.max())
sm = plt.cm.ScalarMappable(cmap=cmap, norm=norm)
sm.set_array([])
for i0 in param_array:
    pmod = eft.model_P_tracer(k_mod, b1, b2, i0, R2, redshift)
    axs[1,0].loglog(k_mod, pmod*k_mod**3/2/np.pi**2, color=cmap(norm(i0)))
cbar = fig.colorbar(sm, ax=axs[1,0], orientation='vertical', pad=0.01)
axs[1,1].set_title('$R2$')
param_array = np.linspace(-20,20,10)
norm = plt.Normalize(vmin=param_array.min(), vmax=param_array.max())
sm = plt.cm.ScalarMappable(cmap=cmap, norm=norm)
sm.set_array([])
for i0 in param_array:
    pmod = eft.model_P_tracer(k_mod, b1, b2, bG2, i0, redshift)
    axs[1,1].loglog(k_mod, pmod*k_mod**3/2/np.pi**2, color=cmap(norm(i0)))
cbar = fig.colorbar(sm, ax=axs[1,1], orientation='vertical', pad=0.01)
for ax in axs.flatten():
    ax.set_xlabel(f'$k [h/\mathrm{{Mpc}}]$')
    ax.set_ylabel(f'$P(k)$')
plt.tight_layout()
plt.show()

## Simulating Gaussian Random Fields (GRF)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import toolscosmo as tcm
import tools21cm as t2c

def simulate_delta_GRF(Om=0.315, Ob=0.049, Or=5.4e-05, As=2.089e-09, h0=0.673, ns=0.963, iSeed=42, grid_size=128, box_size=500):
    """
    Generate Gaussian Random Field (GRF) from matter power spectrum.

    Parameters:
    -----------
    Om : float
        Matter density parameter.
    Ob : float
        Baryon density parameter.
    Or : float
        Radiation density parameter.
    As : float
        Amplitude of the primordial power spectrum.
    h0 : float
        Hubble parameter.
    ns : float
        Spectral index of the primordial power spectrum.
    iSeed : int, optional
        Random seed for reproducibility (default is 42).
    grid_size : int, optional
        Size of the grid for the simulation (default is 128).
    box_size : int, optional
        Size of the simulation box (default is 500 Mpc/h).

    Returns:
    --------
    dict
        A dictionary containing:
        - delta_lin : np.ndarray
            Linear density field in real space.
        - PS : np.ndarray
            Matter power spectrum.
    """
    param = tcm.par()
    param.cosmo.Om = Om
    param.cosmo.Ob = Ob
    param.cosmo.Or = Or
    param.cosmo.Ok = 0.
    param.cosmo.As = As
    param.cosmo.h0 = h0
    param.cosmo.ns = ns
    param.file.ps = 'CAMB' # 'CLASS' 'CLASSemu'
    param.file.ps = tcm.get_Plin(param)
    delta_lin = tcm.generate_gaussian_random_field(grid_size, box_size, param=param, random_seed=iSeed)['delta_lin']
    return {'delta_lin': delta_lin, 'PS': param.file.ps}

def simulate_21cm_GRF(z=9., Om=0.315, Ob=0.049, Or=5.4e-05, As=2.089e-09, h0=0.673, ns=0.963, iSeed=42, grid_size=128, box_size=500):
    """
    Generate Gaussian Random Field (GRF) from matter power spectrum.

    Parameters:
    -----------
    z : float
        Redshift.
    Om : float
        Matter density parameter.
    Ob : float
        Baryon density parameter.
    Or : float
        Radiation density parameter.
    As : float
        Amplitude of the primordial power spectrum.
    h0 : float
        Hubble parameter.
    ns : float
        Spectral index of the primordial power spectrum.
    iSeed : int, optional
        Random seed for reproducibility (default is 42).
    grid_size : int, optional
        Size of the grid for the simulation (default is 128).
    box_size : int, optional
        Size of the simulation box (default is 500 Mpc/h).

    Returns:
    --------
    dict
        A dictionary containing:
        - delta_lin : np.ndarray
            Linear density field in real space.
        - PS : np.ndarray
            Matter power spectrum.
        - dt21 : np.ndarray
            21cm brightness temperature.
    """
    grf_mod = simulate_delta_GRF(Om=Om, Ob=Ob, Or=Or, As=As, h0=h0, ns=ns, iSeed=iSeed, grid_size=grid_size, box_size=box_size)
    dt21 = t2c.mean_dt(z)*(1+grf_mod['delta_lin'])
    grf_mod['dt21'] = dt21
    return grf_mod

if __name__ == "__main__":
    grid_size = 128
    box_size  = 500 #Mpc/h

    Om_list = [0.27, 0.30]
    iSeed_list = [42, 64]

    fig, axs = plt.subplots(2,3,figsize=(13,8))
    xx = np.linspace(0,box_size,grid_size)
    for i,Om in enumerate(Om_list):
        for j,iSeed in enumerate(iSeed_list):
            grf_mod = simulate_21cm_GRF(Om=Om, iSeed=iSeed, grid_size=grid_size, box_size=box_size)
            ps, ks = t2c.power_spectrum_1d(grf_mod['delta_lin'], kbins=15, box_dims=box_size)
            axs[i,j].set_title(r'$\Omega$=%.3f, Random_seed=%d'%(Om,iSeed))
            im = axs[i,j].pcolor(xx, xx, grf_mod['delta_lin'][10])
            axs[i,j].set_xlabel('X [Mpc/h]')
            axs[i,j].set_ylabel('Y [Mpc/h]')
            fig.colorbar(im, ax=axs[i,j])
            axs[i,-1].set_title(r'$\Omega$=%.3f'%(Om))
            axs[i,-1].loglog(ks, ps*ks**3/2/np.pi**2, c=f'C{j}', label=r'Random_seed=%d'%iSeed, zorder=3)
        ps, ks = grf_mod['PS']['P'], grf_mod['PS']['k']
        axs[i,-1].loglog(ks, ps*ks**3/2/np.pi**2, c='k', label=r'CLASSemu', zorder=1)
        axs[i,-1].axis([8e-3,3,3e-3,4])
        axs[i,-1].set_xlabel('k [h/Mpc]')
        axs[i,-1].set_ylabel(r'$\Delta^2$ [h/Mpc]')
    plt.tight_layout()
    plt.show()
            